# Hasil — semua angka dibaca dari file

**Riset Deteksi Fault 4 Sensor Kelembaban**

Notebook ini **tidak menghitung ulang model**. Ia membaca file hasil yang
diekspor notebook penelitian, lalu memeriksa ulang angkanya dengan `assert`.
Kalau file berubah, notebook ini langsung gagal — jadi tidak ada angka yang
diketik manual.

Urutan: performa utama → sensor mana yang rusak → jenis fault → two-stage →
pelatih ANN → XGBoost → satu kalimat statistik → batasan.

In [ ]:
import os, io, glob, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

ASSET_BASE = "https://raw.githubusercontent.com/vousmeevoyez/public-files/main/sidang/assets"
LOCAL_DIRS = ["assets", "sidang/assets", "../sidang/assets"]

def asset_path(rel):
    for base in LOCAL_DIRS:
        p = os.path.join(base, rel)
        if os.path.exists(p):
            return p
    return None

def read_asset(rel):
    p = asset_path(rel)
    if p:
        return pd.read_csv(p)
    r = requests_get(f"{ASSET_BASE}/{rel}")
    return pd.read_csv(io.BytesIO(r))

def requests_get(url):
    import requests
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return r.content

def show_png(rel, width=900):
    p = asset_path(rel)
    if p:
        display(Image(filename=p, width=width))
    else:
        display(Image(data=requests_get(f"{ASSET_BASE}/{rel}"), width=width))

print("Aset siap dibaca dari:", "folder lokal" if asset_path("11/11_rq4_performa_ann.csv") else "GitHub")

## 1 — Performa utama: EDM-Fuzzy vs JSD-Fuzzy

15 sel = 5 skenario × 3 ukuran window (200, 700, 1000 sampel). Selisih
dihitung JSD − EDM; positif berarti JSD lebih tinggi.

In [ ]:
rq4 = read_asset("11/11_rq4_performa_ann.csv")
p = rq4.pivot_table(index=["N_window", "Scenario"], columns="Method", values="F1").reset_index()
p["selisih"] = p["JSD-Fuzzy"] - p["EDM-Fuzzy"]
p["menang"] = np.where(p["selisih"] > 0, "JSD", "EDM")
p["JSD-Fuzzy"] = p["JSD-Fuzzy"].round(3)
p["EDM-Fuzzy"] = p["EDM-Fuzzy"].round(3)
p["selisih"] = p["selisih"].round(3)

n_win = int((p["selisih"] > 0).sum())
mean_diff = float(p["selisih"].mean())
assert n_win == 11, n_win
assert abs(mean_diff - 0.028) < 0.001, mean_diff

print(f"JSD menang {n_win} dari {len(p)} sel | rata-rata selisih {mean_diff:+.3f}")
display(p.to_string(index=False))

In [ ]:
show_png("11/11_rq4_performa_N700.png", 950)
show_png("11/11_rq4_confusion_matrix_N700.png", 760)

**Cara membaca.**

- JSD menang **11/15**, tetapi tidak merata: N = 700 menang 5/5, sedangkan
  N = 200 dan N = 1000 masing-masing 3/5.
- Selisih terbesar di S1 dengan window panjang (**+0,178**); di S3 beberapa sel
  justru negatif.
- Artinya keunggulan JSD **bergantung kondisi**, bukan otomatis di semua kasus.
  Kalimat yang aman: "konsisten lebih tinggi secara rata-rata".

## 2 — Sensor mana yang rusak?

Setelah model menebak ada fault, rantai kedua menebak sensor mana. Angka di
bawah memakai protokol notebook `11` (ANN-LM asli).

In [ ]:
sr = read_asset("11/11_sensor_ringkas.csv")
q = sr.pivot_table(index=["N_window", "Scenario"], columns="Method", values="F1").reset_index()
q["selisih"] = q["JSD-Fuzzy"] - q["EDM-Fuzzy"]

m_edm = float(sr[sr.Method == "EDM-Fuzzy"].F1.mean())
m_jsd = float(sr[sr.Method == "JSD-Fuzzy"].F1.mean())
n_win_s = int((q["selisih"] > 0).sum())
assert abs(m_jsd - 0.675) < 0.002, m_jsd
assert abs(m_edm - 0.634) < 0.002, m_edm
assert n_win_s == 9, n_win_s

print(f"F1 rata-rata: JSD {m_jsd:.3f} vs EDM {m_edm:.3f} | JSD menang {n_win_s}/15")
per_sensor = (sr[sr.Method == "JSD-Fuzzy"]
              .groupby("Scenario").F1.mean().round(3)
              .rename("F1 JSD per skenario").reset_index())
display(per_sensor.to_string(index=False))
show_png("11/11_sensor_f1.png", 900)

**Cara membaca.** Angka di atas adalah **per skenario**, bukan per sensor
fisik: S1 normal vs fault (0,543), S2 fault tunggal (0,622), S3 jumlah fault
(0,635), S4 sensor mana (0,735), S5 semua kondisi (0,841).

Angka pendamping: tebakan **semua empat sensor benar sekaligus** 0,307 (JSD)
vs 0,295 (EDM). Ini lebih ketat daripada F1 rata-rata.

Rincian per **sensor fisik** ada di bagian 3 (notebook `05`).

## 3 — Jenis fault apa, di sensor mana?

Notebook `05` memakai 16 label (4 sensor × 4 jenis fault). Tabel ini menjawab
"bias-nya di sensor mana?" dan "spike-nya di mana?".

In [ ]:
f5 = read_asset("05/cara_b_stage2_summary.csv")
fault = f5[f5["Sudut pandang"] == "per jenis fault"].pivot(
    index="Item", columns="Method", values="F1 rata-rata").round(3)
fault["selisih"] = (fault["JSD-Fuzzy"] - fault["EDM-Fuzzy"]).round(3)
assert abs(float(fault.loc["bias", "JSD-Fuzzy"]) - 0.448) < 0.001
assert abs(float(fault.loc["spike", "JSD-Fuzzy"]) - 0.601) < 0.001
display(fault.reset_index().to_string(index=False))

per_sensor = f5[f5["Sudut pandang"] == "per sensor"].pivot(
    index="Item", columns="Method", values="F1 rata-rata").round(3)
per_sensor["selisih"] = (per_sensor["JSD-Fuzzy"] - per_sensor["EDM-Fuzzy"]).round(3)
assert abs(float(per_sensor.loc["S3", "JSD-Fuzzy"]) - 0.760) < 0.001
assert abs(float(per_sensor.loc["S4", "JSD-Fuzzy"]) - 0.669) < 0.001
display(per_sensor.reset_index().to_string(index=False))
show_png("05/cara_b_heatmap_f1.png", 900)

**Cara membaca.** Per **jenis fault**, urutan kesulitan: **bias 0,448 <
spike 0,601 < drift 0,868 < hardware 0,917**. Bias dan spike adalah titik
terlemah kedua metode; di situ JSD menolong paling besar (+0,306 dan +0,410).
Hardware sudah mudah dikenali keduanya, jadi selisihnya praktis nol.

Per **sensor fisik** (JSD): S3 0,760 > S1 0,724 > S2 0,682 > S4 0,669. S4
paling sulit karena banyak periode tanah kering.

## 4 — Two-stage: klasifikasi dulu, sensor kemudian

Notebook `04` menguji rantai penuh: dari 17 kelas, ambil prediksi, lalu tebak
sensor. Tabel pertama per tingkat kesulitan; tabel kedua per jumlah fault yang
bercampur.

In [ ]:
t2 = read_asset("04/tahap2_ringkas_per_tugas.csv")
t3 = t2[t2.Tugas == "T3_17kelas"].set_index("Metode")["F1"]
assert abs(float(t3["JSD-Fuzzy"]) - 0.902) < 0.001
assert abs(float(t3["EDM-Fuzzy"]) - 0.846) < 0.001
display(t2.round(3).to_string(index=False))

jf = read_asset("04/tahap2_per_jumlah_fault.csv")
all4 = jf[jf["Jumlah fault"] == 4].set_index("Metode")["Semua-4-benar"]
assert abs(float(all4["JSD-Fuzzy"]) - 0.920) < 0.001
assert abs(float(all4["EDM-Fuzzy"]) - 0.800) < 0.001
display(jf.round(3).to_string(index=False))

**Cara membaca.** Tahap-2 paling sulit pada 17 kelas: F1 **0,902 (JSD)** vs
0,846 (EDM). Kalau keempat sensor rusak sekaligus, peluang menebak keempatnya
benar **0,920 vs 0,800**. Nilai ini lebih rendah daripada F1 per sensor, dan
itu wajar: semua harus benar, bukan rata-rata.

## 5 — Empat algoritma pelatih ANN

Notebook `09` membandingkan LM (paper), Bayesian regularization (BR), GA, dan
Adam pada fold yang sama.

In [ ]:
rk = read_asset("09/09_rq2_peringkat_pelatih.csv").set_index("Trainer").round(3)
assert abs(float(rk.loc["BR", "mean_F1"]) - 0.429) < 0.001
assert abs(float(rk.loc["LM", "mean_F1"]) - 0.433) < 0.001
display(rk.reset_index().to_string(index=False))
show_png("09/09_rq2_performa.png", 900)

**Cara membaca.** BR dan LM setara (0,429 vs 0,433; p = 0,81), keduanya di atas
GA (0,306) dan Adam (0,268). Jadi pilihan LM pada paper tetap masuk akal,
tetapi bukan satu-satunya yang bisa bekerja.

## 6 — ANN vs XGBoost: batasnya di mana?

Notebook `06` menguji apakah keunggulan datang dari fitur atau dari classifier.

In [ ]:
x6 = read_asset("06/xgb_perscenario_results.csv")
pv = x6.pivot_table(index="Scenario", columns="FeatureSet", values="F1_macro").round(3)
avg = pv.mean().round(3)
assert abs(float(avg["JSD+Time"]) - 0.750) < 0.002, avg.to_dict()
display(pv.to_string())
display(avg.rename("rata-rata 5 skenario").to_frame().T.to_string())

**Cara membaca.** Rata-rata 5 skenario: **JSD+Time 0,750**, Time saja 0,701,
EDM+Time 0,684, JSD saja 0,583, EDM saja 0,418. Artinya dua hal berkontribusi:
fitur JSD lebih baik dari EDM, dan XGBoost memanfaatkan fitur waktu dengan
baik. Di skenario tersulit (S5) angkanya 0,608 — bukan 0,750.

Jangan membandingkan 0,750 (XGBoost, rata-rata) dengan 0,433 (ANN, protokol
notebook 09). Protokol dan set fiturnya berbeda.

## 7 — Satu kalimat statistik (hanya satu)

Uji berpasangan pada notebook `11`: N = 700 **p = 0,027**; gabungan tiga ukuran
window **p = 0,076** (Wilcoxon 0,107). Artinya: selisih +0,028 konsisten, tetapi
belum kuat secara gabungan. Rincian sengaja tidak ditampilkan di edisi sidang;
tersedia di lampiran privat `11` §11.

**Yang tidak diklaim:** "JSD terbukti lebih baik". **Yang diklaim:** "JSD
konsisten lebih tinggi pada 11 dari 15 sel".

## 8 — Batasan yang harus disebut

1. Fault berasal dari **simulasi** dengan rumus tetap, bukan kerusakan lapangan.
2. Keputusan butuh window 16,7 jam → pemantauan harian, **bukan alarm real-time**.
3. Keunggulan JSD tidak merata; di S3 beberapa sel negatif.
4. EDM lebih **stabil** (CV lebih rendah di 56/63 pasangan), walau F1 lebih
   rendah. Ini dilaporkan apa adanya.
5. Drift 1 poin/hari sengaja kecil; deteksi drift jangka pendek masih lemah.

## 9 — Sumber setiap blok

| Blok | Notebook asli | Sel | File |
|---|---|---|---|
| 1 Performa | `11` §10 | 33–40 | `11_rq4_performa_ann.csv` |
| 2 Sensor-ID | `11` §12 | 43–46 | `11_sensor_ringkas.csv` |
| 3 Jenis fault | `05` | 13–19 | `cara_b_stage2_summary.csv` |
| 4 Two-stage | `04` | 15–20 | `tahap2_*.csv` |
| 5 Pelatih | `09` §10 | 29–34 | `09_rq2_peringkat_pelatih.csv` |
| 6 XGBoost | `06` | 36–41 | `xgb_perscenario_results.csv` |
| 7 Statistik | `11` §11 | 41–42 | `11_rq5_ttest.csv` (privat) |

---
**Status:** privat · commit `27737ee` · 2026-10-26 · rujukan: `sidang/MAPPING.md`